# 04 — TGN Training on 3W Dataset

**Goal:** Train and evaluate the Temporal Graph Network on the 3W oil well dataset.

## Key differences from Use Case 1 (synthetic data):
- Real industrial data with genuine noise and missing values
- Multiple wells as graph nodes (generalizable model)
- PRECEDES and CAUSALLY_COUPLED relations used by TGN
- Strict temporal split to prevent data leakage
- Weighted loss for class imbalance (rare events)

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

torch.manual_seed(42)
np.random.seed(42)

PROCESSED_DIR = Path('../../data/processed')
MODELS_DIR    = Path('../../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


PyTorch: 2.9.1+cu128
Device: GPU


## 1. Load Data from Neo4j

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path('/home/obiaggi/3w_processed')
MODELS_DIR    = Path('/home/obiaggi/TKG_Thesis/models')
MODELS_DIR.mkdir(exist_ok=True, parents=True)

SENSOR_COLS = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'QGL', 'T-PDG']
ROWS_PER_CLASS = 10000  # 10k per class = 100k total

CLASS_NAMES = {
    0:'Normal', 1:'BSW increase', 2:'DHSV closure', 3:'Severe slugging',
    4:'Flow instability', 5:'Productivity loss', 6:'Quick restriction',
    7:'Scaling', 8:'Hydrate', 9:'Other'
}

dfs = []
for type_num in range(10):
    f = PROCESSED_DIR / f'type_{type_num}.parquet'
    if not f.exists():
        print(f'  Missing type_{type_num} — skipping')
        continue
    d = pd.read_parquet(f)
    if 'event_type' in d.columns and 'class' not in d.columns:
        d = d.rename(columns={'event_type': 'class'})
    if 'instance_id' not in d.columns:
        d['instance_id'] = f'WELL-{type_num:02d}'
    available = [c for c in SENSOR_COLS if c in d.columns]
    d = d.dropna(subset=available, how='all')
    if len(d) > ROWS_PER_CLASS:
        d = d.sample(ROWS_PER_CLASS, random_state=42)
    d['is_anomaly'] = (d['class'].fillna(0) > 0).astype(int)
    d['type_num'] = type_num
    dfs.append(d)
    print(f'  type_{type_num} ({CLASS_NAMES[type_num]:<20}): {len(d):>6,} rows | anomaly: {d["is_anomaly"].mean()*100:.0f}%')

df = pd.concat(dfs, ignore_index=True)
available_sensors = [c for c in SENSOR_COLS if c in df.columns]
df['value']     = df[available_sensors].fillna(0).mean(axis=1)
df['sensor_id'] = 'SENSOR'
df['well_id']   = df['instance_id'].astype(str).fillna('WELL-00')

print(f'\nTotal: {len(df):,} | Normal: {(df["is_anomaly"]==0).sum():,} | Anomaly: {(df["is_anomaly"]==1).sum():,}')
print(f'Anomaly rate: {df["is_anomaly"].mean()*100:.1f}%')


## 2. Prepare TGN Data

In [ ]:
from sklearn.model_selection import train_test_split as sk_split
from sklearn.preprocessing import StandardScaler

def prepare_tgn_data(df: pd.DataFrame):
    if df.empty:
        return None, None, 0

    df = df.copy()
    wells   = df['well_id'].unique().tolist()
    sensors = df['sensor_id'].unique().tolist()
    all_nodes = wells + sensors
    node2idx  = {n: i for i, n in enumerate(all_nodes)}

    scaler = StandardScaler()
    df['value_norm']   = scaler.fit_transform(df[['value']].fillna(0))
    df['delta_t_norm'] = 0.0  # flat parquet has no real timestamps

    # STRATIFIED SPLIT 70/30 — fixes the temporal split imbalance
    idx = np.arange(len(df))
    train_idx, test_idx = sk_split(
        idx, test_size=0.3, random_state=42, stratify=df['is_anomaly'].values)
    train_df = df.iloc[train_idx]
    test_df  = df.iloc[test_idx]

    def to_tensors(d):
        import torch
        src = torch.tensor([node2idx.get(w, 0) for w in d['well_id']], dtype=torch.long)
        dst = torch.tensor([node2idx.get(s, 0) for s in d['sensor_id']], dtype=torch.long)
        ef  = torch.tensor(d[['value_norm','delta_t_norm']].values, dtype=torch.float32)
        dt  = torch.tensor(d['delta_t_norm'].values, dtype=torch.float32).unsqueeze(1)
        y   = torch.tensor(d['is_anomaly'].values.astype(float), dtype=torch.float32)
        return src, dst, ef, dt, y

    print(f'Train: {len(train_df):,} | Normal: {(train_df["is_anomaly"]==0).sum():,} | Anomaly: {(train_df["is_anomaly"]==1).sum():,}')
    print(f'Test:  {len(test_df):,}  | Normal: {(test_df["is_anomaly"]==0).sum():,}  | Anomaly: {(test_df["is_anomaly"]==1).sum():,}')
    print(f'Anomaly rate — train: {train_df["is_anomaly"].mean()*100:.1f}% | test: {test_df["is_anomaly"].mean()*100:.1f}%')
    return to_tensors(train_df), to_tensors(test_df), len(all_nodes)

train_data, test_data, num_nodes = prepare_tgn_data(df)


## 3. Train TGN

In [4]:
# Import TGN from src/models
try:
    from models.tgn import TGN, train_tgn, evaluate_tgn
    print('✅ TGN imported from src/models')
except ImportError:
    print('⚠️  TGN not found in src/models — copy TGN-anomaly_detection.py there first')

if not df.empty and train_data is not None:
    model = TGN(
        num_nodes=num_nodes,
        memory_dim=64,    # larger than UC1 — more complex data
        message_dim=64,
        embed_dim=64,
        edge_dim=2,
    )
    print(f'TGN parameters: {sum(p.numel() for p in model.parameters()):,}')

    print('\n🏋️  Training TGN on 3W dataset...')
    train_tgn(model, train_data, epochs=5, batch_size=512)

    print('\n📊 Evaluation:')
    evaluate_tgn(model, test_data)

    # Save checkpoint
    checkpoint_path = MODELS_DIR / 'tgn_3w_checkpoint.pt'
    torch.save({
        'model_state': model.state_dict(),
        'num_nodes': num_nodes,
        'dataset': '3W',
        'epochs': 5,
    }, checkpoint_path)
    print(f'\n✅ Checkpoint saved: {checkpoint_path}')
else:
    print('⚠️  Load data first')

✅ TGN imported from src/models
TGN parameters: 65,537

🏋️  Training TGN on 3W dataset...
  Epoch 1/5 — loss: 1.7338
  Epoch 2/5 — loss: 1.1503
  Epoch 3/5 — loss: 2.2902
  Epoch 4/5 — loss: 0.5534
  Epoch 5/5 — loss: 0.8318

📊 Evaluation:

📈 Risultati: TGN (Temporal Graph Network)
              precision    recall  f1-score   support

      Normal      1.000     1.000     1.000     29998
     Anomaly      0.000     0.000     0.000         2

    accuracy                          1.000     30000
   macro avg      0.500     0.500     0.500     30000
weighted avg      1.000     1.000     1.000     30000

  AUC-ROC: 0.6108

✅ Checkpoint saved: /home/obiaggi/TKG_Thesis/models/tgn_3w_checkpoint.pt


/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/obiaggi/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
